# Session 2: Open-Source LLMs with Ollama & Embeddings
**Duration:** 1.5 Hours | **Day 1** | **Hands-On Workshop**

---

## Learning Objectives
By the end of this session, participants will:
1. Understand how open-source LLMs differ from proprietary models
2. Work with Ollama to manage and run local models
3. Understand tokenization and how LLMs process text
4. Generate and use vector embeddings
5. Build a practical "Resume Analyzer" mini-project

---

## Prerequisites
- Ollama running with models: `llama3.2`, `nomic-embed-text`
- Packages: `pip install langchain langchain-ollama sentence-transformers numpy scikit-learn`

---
## Part 1: Open-Source LLM Landscape (Theory - 15 min)

### 1.1 Why Open-Source LLMs Matter

| Aspect | Proprietary (GPT-4, Claude) | Open-Source (LLaMA, Mistral) |
|--------|---------------------------|-----------------------------|
| **Cost** | Pay-per-token API calls | Free to run locally |
| **Privacy** | Data sent to external servers | Data stays on your machine |
| **Customization** | Limited to prompting | Can fine-tune weights |
| **Latency** | Depends on internet/API | Depends on local hardware |
| **Transparency** | Black box | Full model weights available |
| **Offline Use** | No | Yes |

### 1.2 Key Open-Source Model Families

- **LLaMA (Meta)**: LLaMA 2, LLaMA 3 -- General purpose, strong reasoning
- **Mistral**: Mistral 7B, Mixtral 8x7B -- Efficient, fast, good for production
- **Gemma (Google)**: Gemma 2B, 7B -- Lightweight, good on consumer hardware
- **Phi (Microsoft)**: Phi-2, Phi-3 -- Small but powerful models
- **Qwen (Alibaba)**: Multi-lingual, strong in Asian languages

### 1.3 Ollama: Your Local LLM Runtime

Ollama makes it trivially easy to run open-source LLMs locally:
```bash
ollama pull llama3.2       # Download a model
ollama run llama3.2        # Interactive chat
ollama list                # See all downloaded models
ollama show llama3.2       # Model details
```

Under the hood, Ollama:
- Downloads quantized model weights (GGUF format)
- Runs inference using llama.cpp
- Exposes a REST API on `http://localhost:11434`

In [ ]:
# 1.4 Exploring Ollama API -- List models and their details
import requests
import json

# List all available models
response = requests.get("http://localhost:11434/api/tags")
models = response.json()

print("=== Downloaded Models ===")
for model in models.get('models', []):
    size_gb = model.get('size', 0) / (1024**3)
    print(f"  Model: {model['name']}")
    print(f"  Size: {size_gb:.2f} GB")
    print(f"  Modified: {model.get('modified_at', 'N/A')}")
    print()

---
## Part 2: Tokenization -- How LLMs See Text (Theory + Hands-On - 20 min)

### 2.1 What is Tokenization?

LLMs do not process raw text. They process **tokens** -- subword units derived from the text.

```
"Hello, how are you?" 
    --> ["Hello", ",", " how", " are", " you", "?"]
    --> [15339, 11, 1268, 527, 499, 30]
```

**Key Points:**
- Common words = fewer tokens ("the" = 1 token)
- Rare words = more tokens ("pneumonoultramicroscopicsilicovolcanoconiosis" = many tokens)
- Spaces/punctuation count as tokens
- **Context window** = maximum number of tokens the model can process at once
  - LLaMA 3.2: 128K tokens
  - GPT-4: 128K tokens

In [ ]:
# 2.2 Hands-On: Exploring Tokenization with Transformers library
from transformers import AutoTokenizer

# Load a tokenizer (GPT-2 tokenizer is openly available)
tokenizer = AutoTokenizer.from_pretrained("gpt2")

# Tokenize different texts
texts = [
    "Hello, world!",
    "Artificial Intelligence is transforming the world.",
    "The quick brown fox jumps over the lazy dog.",
    "pneumonoultramicroscopicsilicovolcanoconiosis",
    "Bangalore is the Silicon Valley of India.",
]

print("=== Tokenization Examples ===")
for text in texts:
    tokens = tokenizer.encode(text)
    decoded_tokens = [tokenizer.decode([t]) for t in tokens]
    print(f"\nText: \"{text}\"")
    print(f"  Token count: {len(tokens)}")
    print(f"  Token IDs: {tokens}")
    print(f"  Tokens: {decoded_tokens}")

In [ ]:
# 2.3 Why Tokenization Matters -- Cost and Context

# Simulate token counting for different content types
samples = {
    "Short query": "What is machine learning?",
    "Medium paragraph": """Machine learning is a subset of artificial intelligence that
    enables systems to learn from data without being explicitly programmed. It uses
    algorithms to identify patterns in data and make predictions or decisions.""",
    "Code snippet": """def quicksort(arr):\n    if len(arr) <= 1:\n        return arr\n
    pivot = arr[len(arr) // 2]\n    left = [x for x in arr if x < pivot]\n
    middle = [x for x in arr if x == pivot]\n    right = [x for x in arr if x > pivot]\n
    return quicksort(left) + middle + quicksort(right)""",
    "JSON data": '{"name": "John", "age": 30, "skills": ["Python", "ML", "Docker"]}'
}

print("=== Token Counts by Content Type ===")
for label, text in samples.items():
    tokens = tokenizer.encode(text)
    chars = len(text)
    ratio = chars / len(tokens)
    print(f"  {label}:")
    print(f"    Characters: {chars} | Tokens: {len(tokens)} | Chars/Token: {ratio:.1f}")

---
## Part 3: Vector Embeddings (Theory + Hands-On - 25 min)

### 3.1 What are Embeddings?

Embeddings convert text into **dense numerical vectors** that capture semantic meaning.

```
"king"   --> [0.2, 0.8, -0.1, 0.5, ...] (768 dimensions)
"queen"  --> [0.3, 0.7, -0.2, 0.6, ...] (similar to king!)
"banana" --> [-0.5, 0.1, 0.9, -0.3, ...] (very different)
```

**Key Properties:**
- Similar meanings = Close vectors (cosine similarity ~ 1)
- Different meanings = Distant vectors (cosine similarity ~ 0)
- Classic example: king - man + woman ~ queen

**Why Embeddings Matter:**
- Foundation for semantic search
- Required for RAG (Retrieval Augmented Generation)
- Enable similarity matching, clustering, recommendations

In [ ]:
# 3.2 Generate Embeddings using Ollama
import numpy as np

def get_embedding(text, model="nomic-embed-text"):
    """Get embedding from Ollama."""
    response = requests.post(
        "http://localhost:11434/api/embed",
        json={"model": model, "input": text}
    )
    return np.array(response.json()["embeddings"][0])

# Generate embeddings for sample texts
text1 = "Python is a programming language"
text2 = "Java is used for software development"
text3 = "The weather today is sunny and warm"

emb1 = get_embedding(text1)
emb2 = get_embedding(text2)
emb3 = get_embedding(text3)

print(f"Embedding dimensions: {emb1.shape[0]}")
print(f"\nFirst 10 values of embedding for '{text1}':")
print(f"  {emb1[:10]}")

In [ ]:
# 3.3 Cosine Similarity -- Measuring Semantic Closeness
from sklearn.metrics.pairwise import cosine_similarity

def compute_similarity(emb_a, emb_b):
    """Compute cosine similarity between two embeddings."""
    return cosine_similarity([emb_a], [emb_b])[0][0]

# Compare similarities
sim_12 = compute_similarity(emb1, emb2)  # Programming vs Programming
sim_13 = compute_similarity(emb1, emb3)  # Programming vs Weather
sim_23 = compute_similarity(emb2, emb3)  # Programming vs Weather

print("=== Cosine Similarity Results ===")
print(f"  '{text1}' vs '{text2}': {sim_12:.4f}  (Both about programming - HIGH)")
print(f"  '{text1}' vs '{text3}': {sim_13:.4f}  (Programming vs Weather - LOW)")
print(f"  '{text2}' vs '{text3}': {sim_23:.4f}  (Programming vs Weather - LOW)")

In [ ]:
# 3.4 Semantic Search Demo -- Finding the most relevant text

# Knowledge base
documents = [
    "Python is widely used for data science and machine learning applications.",
    "Docker containers help in deploying applications consistently across environments.",
    "Neural networks are inspired by the structure of the human brain.",
    "Kubernetes orchestrates containerized applications at scale.",
    "Transfer learning allows using pre-trained models for new tasks.",
    "REST APIs enable communication between different software systems.",
    "Gradient descent is an optimization algorithm used in training neural networks.",
    "Git is a version control system for tracking code changes.",
]

# Generate embeddings for all documents
doc_embeddings = [get_embedding(doc) for doc in documents]

# Search function
def semantic_search_demo(query, top_k=3):
    """Find the most semantically similar documents to a query."""
    query_emb = get_embedding(query)
    similarities = [compute_similarity(query_emb, doc_emb) for doc_emb in doc_embeddings]

    # Sort by similarity (descending)
    ranked = sorted(enumerate(similarities), key=lambda x: x[1], reverse=True)

    print(f"Query: \"{query}\"\n")
    print("Top results:")
    for rank, (idx, score) in enumerate(ranked[:top_k], 1):
        print(f"  {rank}. [{score:.4f}] {documents[idx]}")
    print()

# Test different queries
semantic_search_demo("How do I train a deep learning model?")
semantic_search_demo("How to deploy my application?")
semantic_search_demo("What programming language should I learn for AI?")

---
## Part 4: Fine-Tuning vs Prompting (Theory - 10 min)

### When to Use Each Approach

| Approach | When to Use | Effort | Cost |
|----------|-------------|--------|------|
| **Zero-Shot Prompting** | Simple tasks, general knowledge | Low | Low (no training) |
| **Few-Shot Prompting** | Pattern-following tasks | Low | Low |
| **RAG (Retrieval)** | Domain-specific knowledge | Medium | Medium (vector DB) |
| **Fine-Tuning (LoRA)** | Specific style/behavior change | High | High (GPU needed) |
| **Full Fine-Tuning** | Complete domain adaptation | Very High | Very High |

### Decision Framework:
```
Need domain knowledge? --> RAG
Need behavior change? --> Fine-tuning
Need format control? --> Prompt Engineering
Need all of the above? --> RAG + Fine-tuned model + Good prompts
```

For most production use cases, **RAG + Prompt Engineering** covers 80% of needs.

---
## Part 5: Mini Project -- Resume Analyzer (Hands-On - 20 min)

Build a system that:
1. Takes a resume text as input
2. Extracts key information (skills, experience, education)
3. Matches against a job description using embeddings
4. Provides a fit score and recommendations

In [ ]:
# 5.1 Resume Analyzer -- Setup
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import json

# Sample resume
resume_text = """
RAHUL KUMAR
Software Engineer | Bangalore, India
Email: rahul.kumar@email.com | Phone: +91-9876543210

SUMMARY:
3 years of experience in backend development with Python and Java.
Passionate about building scalable microservices and working with cloud technologies.

SKILLS:
- Programming: Python, Java, JavaScript
- Frameworks: Django, Spring Boot, FastAPI
- Databases: PostgreSQL, MongoDB, Redis
- Cloud: AWS (EC2, S3, Lambda), Docker, Kubernetes
- Tools: Git, Jenkins, Terraform

EXPERIENCE:
Software Engineer at TechCorp (2021-Present)
- Built RESTful APIs serving 1M+ requests/day
- Migrated monolith to microservices architecture
- Implemented CI/CD pipeline reducing deployment time by 60%

Junior Developer at StartupXYZ (2020-2021)
- Developed backend services using Django
- Wrote unit tests achieving 85% code coverage

EDUCATION:
B.Tech in Computer Science, VTU (2016-2020), CGPA: 8.5/10
"""

# Sample job description
job_description = """
Senior Backend Engineer at AI Solutions Inc.

Requirements:
- 3+ years of experience in backend development
- Strong proficiency in Python
- Experience with FastAPI or Django
- Knowledge of Docker and Kubernetes
- Experience with machine learning pipelines (preferred)
- Familiarity with LLMs and AI frameworks (preferred)
- Strong problem-solving skills
- Experience with cloud platforms (AWS/GCP)
"""

print("Resume and Job Description loaded successfully!")

In [ ]:
# 5.2 Step 1: Extract structured information from resume using LLM

chat_llm = ChatOllama(model="llama3.2", temperature=0)

extraction_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a resume parsing expert. Extract information from resumes
    and return ONLY valid JSON. No additional text."""),
    ("human", """Extract the following information from this resume and return as JSON:

Resume:
{resume}

Return JSON with this exact structure:
{{
    "name": "",
    "location": "",
    "total_experience_years": 0,
    "current_role": "",
    "skills": {{
        "programming_languages": [],
        "frameworks": [],
        "databases": [],
        "cloud_devops": [],
        "tools": []
    }},
    "education": "",
    "key_achievements": []
}}""")
])

chain = extraction_prompt | chat_llm | StrOutputParser()
extracted = chain.invoke({"resume": resume_text})

print("=== Extracted Resume Data ===")
print(extracted)

In [ ]:
# 5.3 Step 2: Compute Semantic Similarity between Resume and Job Description

# Get embeddings for resume and job description
resume_embedding = get_embedding(resume_text)
job_embedding = get_embedding(job_description)

# Overall fit score
overall_similarity = compute_similarity(resume_embedding, job_embedding)
print(f"=== Overall Resume-Job Fit Score ===")
print(f"  Cosine Similarity: {overall_similarity:.4f}")
print(f"  Fit Percentage: {overall_similarity * 100:.1f}%")
print()

# Skill-level matching -- compare each required skill against the resume
required_skills = [
    "Python programming",
    "FastAPI or Django framework",
    "Docker and Kubernetes",
    "Machine learning pipelines",
    "LLMs and AI frameworks",
    "AWS or GCP cloud platform",
    "Backend development experience"
]

print("=== Skill-Level Match ===")
for skill in required_skills:
    skill_emb = get_embedding(skill)
    score = compute_similarity(skill_emb, resume_embedding)
    match_level = "STRONG" if score > 0.5 else "MODERATE" if score > 0.35 else "WEAK"
    bar = "|" * int(score * 50)
    print(f"  {skill:40s} [{score:.3f}] {match_level:8s} {bar}")

In [ ]:
# 5.4 Step 3: Generate Recommendations using LLM

recommendation_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a career advisor specializing in tech roles.
    Analyze resume against job requirements and provide actionable advice."""),
    ("human", """Compare this resume against the job description and provide:

1. MATCH ANALYSIS: What requirements does the candidate meet?
2. GAP ANALYSIS: What requirements are the candidate missing?
3. RECOMMENDATIONS: Top 3 specific actions to improve their chances
4. OVERALL VERDICT: Fit rating (Strong Match / Moderate Match / Weak Match)

Resume:
{resume}

Job Description:
{job_description}

Be specific and actionable in your recommendations.""")
])

chain = recommendation_prompt | chat_llm | StrOutputParser()
recommendations = chain.invoke({
    "resume": resume_text,
    "job_description": job_description
})

print("=== Career Recommendations ===")
print(recommendations)

In [ ]:
# 5.5 Putting it all together -- Complete Resume Analyzer function

def analyze_resume(resume: str, job_desc: str):
    """Complete resume analysis pipeline."""
    print("=" * 60)
    print("         RESUME ANALYZER -- AI-Powered Report")
    print("=" * 60)

    # Step 1: Semantic Similarity
    resume_emb = get_embedding(resume)
    job_emb = get_embedding(job_desc)
    fit_score = compute_similarity(resume_emb, job_emb)

    print(f"\n[SECTION 1] OVERALL FIT SCORE: {fit_score * 100:.1f}%")
    print("-" * 40)

    if fit_score > 0.6:
        print("  VERDICT: STRONG MATCH -- Proceed with application")
    elif fit_score > 0.4:
        print("  VERDICT: MODERATE MATCH -- Worth applying with improvements")
    else:
        print("  VERDICT: WEAK MATCH -- Consider upskilling first")

    # Step 2: LLM Analysis
    print(f"\n[SECTION 2] DETAILED ANALYSIS")
    print("-" * 40)

    analysis_prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a precise career advisor. Be concise and actionable."),
        ("human", """Given this resume and job description, provide:
        1. Top 3 strengths for this role
        2. Top 3 gaps to address
        3. Top 3 action items

        Resume: {resume}
        Job: {job_desc}

        Be brief -- max 2 lines per point.""")
    ])

    chain = analysis_prompt | chat_llm | StrOutputParser()
    analysis = chain.invoke({"resume": resume, "job_desc": job_desc})
    print(analysis)

    print("\n" + "=" * 60)
    print("             END OF REPORT")
    print("=" * 60)

# Run the complete analyzer
analyze_resume(resume_text, job_description)

---
## Exercises

### Exercise 1: Embedding Explorer
Create a list of 10 sentences from different domains (tech, sports, cooking, etc.). Generate embeddings and compute all pairwise similarities. Which pairs are most/least similar?

### Exercise 2: Enhanced Resume Analyzer
Extend the resume analyzer to:
- Accept multiple job descriptions and rank them by fit
- Generate a tailored cover letter based on the best match

### Exercise 3: Prompt Comparison
Use the same resume with different system prompts (strict recruiter vs friendly advisor). Compare how the tone and recommendations change.

---

## Summary

### Key Takeaways:
1. **Open-source LLMs** (via Ollama) provide privacy, cost-efficiency, and customization
2. **Tokenization** is how LLMs process text -- token counts affect cost and context limits
3. **Embeddings** are dense vector representations that capture semantic meaning
4. **Cosine similarity** measures how semantically related two pieces of text are
5. **Semantic search** uses embeddings to find relevant content beyond keyword matching
6. For most use cases, **RAG + Prompt Engineering** is more practical than fine-tuning

### Next Session: Building a RAG Pipeline
We will use these embedding concepts to build a full Retrieval Augmented Generation system with LangChain and FAISS.